# Random Forest Training

Baseline training and fine-tuning workflow for the Random Forest model used in the `distraction_prediction` module.

## Imports

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)


## Paths

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'distraction_prediction' / 'data' / 'processed' / 'windows').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing distraction_prediction/data/processed/windows')


PROJECT_ROOT = find_project_root()
WINDOWS_DIR = PROJECT_ROOT / 'distraction_prediction' / 'data' / 'processed' / 'windows'
SAVE_DIR = PROJECT_ROOT / 'distraction_prediction' / 'models' / 'saved_models' / 'baselines'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Windows dir  :', WINDOWS_DIR)
print('Save dir     :', SAVE_DIR)


Project root : e:\SDPPS
Windows dir  : e:\SDPPS\distraction_prediction\data\processed\windows
Save dir     : e:\SDPPS\distraction_prediction\models\saved_models\baselines


## Helper Functions

In [3]:
def load_window_splits():
    return {
        'X_train': np.load(WINDOWS_DIR / 'X_train.npy'),
        'y_train': np.load(WINDOWS_DIR / 'y_train.npy'),
        'X_val': np.load(WINDOWS_DIR / 'X_val.npy'),
        'y_val': np.load(WINDOWS_DIR / 'y_val.npy'),
        'X_test': np.load(WINDOWS_DIR / 'X_test.npy'),
        'y_test': np.load(WINDOWS_DIR / 'y_test.npy'),
    }


def flatten_windows(X_3d):
    mean_features = np.mean(X_3d, axis=1)
    last_features = X_3d[:, -1, :]
    return np.hstack([mean_features, last_features]).astype(np.float32)


def prepare_flattened_splits():
    data = load_window_splits()
    return {
        'X_train': flatten_windows(data['X_train']),
        'y_train': data['y_train'],
        'X_val': flatten_windows(data['X_val']),
        'y_val': data['y_val'],
        'X_test': flatten_windows(data['X_test']),
        'y_test': data['y_test'],
    }


def combine_train_and_val(data):
    X_trainval = np.concatenate([data['X_train'], data['X_val']], axis=0)
    y_trainval = np.concatenate([data['y_train'], data['y_val']], axis=0)
    return X_trainval, y_trainval


def evaluate_binary_classifier(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    matrix = confusion_matrix(y, predictions)
    return {
        'accuracy': accuracy_score(y, predictions),
        'f1': f1_score(y, predictions, zero_division=0),
        'auc': roc_auc_score(y, probabilities) if len(np.unique(y)) > 1 else 0.0,
        'confusion_matrix': matrix.tolist(),
        'classification_report': classification_report(
            y,
            predictions,
            target_names=['Focused', 'Distracted'],
            zero_division=0,
        ),
    }


def print_dataset_summary(data):
    print('Loaded flattened datasets')
    print(f"  Train: {data['X_train'].shape} | Positives: {int(data['y_train'].sum())}")
    print(f"  Val  : {data['X_val'].shape} | Positives: {int(data['y_val'].sum())}")
    print(f"  Test : {data['X_test'].shape} | Positives: {int(data['y_test'].sum())}")


def print_metrics(title, metrics):
    cm = metrics['confusion_matrix']
    print('\n' + '=' * 44)
    print(title)
    print('=' * 44)
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"F1 Score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['auc']:.4f}")
    print('Confusion Matrix:')
    print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
    print(f"  FN={cm[1][0]}  TP={cm[1][1]}")
    print('=' * 44)
    print(metrics['classification_report'])


def save_model_and_results(model, model_name, results_name, results):
    joblib.dump(model, SAVE_DIR / model_name)
    serializable_results = {}
    for key, value in results.items():
        if isinstance(value, (np.floating, float)):
            serializable_results[key] = round(float(value), 4)
        else:
            serializable_results[key] = value
    with open(SAVE_DIR / results_name, 'w', encoding='utf-8') as file:
        json.dump(serializable_results, file, indent=2)


## Load Prepared Window Data

In [4]:
data = prepare_flattened_splits()
print_dataset_summary(data)


Loaded flattened datasets
  Train: (51371, 48) | Positives: 29477
  Val  : (6421, 48) | Positives: 3685
  Test : (6422, 48) | Positives: 3685


## Baseline Random Forest

In [5]:
BASELINE_CONFIG = {
    'n_estimators': 300,
    'max_depth': 15,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
}

X_trainval, y_trainval = combine_train_and_val(data)

baseline_model = RandomForestClassifier(**BASELINE_CONFIG)
baseline_model.fit(X_trainval, y_trainval)

baseline_metrics = evaluate_binary_classifier(baseline_model, data['X_test'], data['y_test'])
print_metrics('Baseline Random Forest Results', baseline_metrics)



Baseline Random Forest Results
Accuracy : 0.8806
F1 Score : 0.8908
ROC-AUC  : 0.9548
Confusion Matrix:
  TN=2525  FP=212
  FN=555  TP=3130
              precision    recall  f1-score   support

     Focused       0.82      0.92      0.87      2737
  Distracted       0.94      0.85      0.89      3685

    accuracy                           0.88      6422
   macro avg       0.88      0.89      0.88      6422
weighted avg       0.89      0.88      0.88      6422



In [6]:
save_model_and_results(
    model=baseline_model,
    model_name='random_forest.joblib',
    results_name='random_forest_results.json',
    results=baseline_metrics,
)


## Fine-Tuned Random Forest

In [7]:
SEARCH_CONFIGS = [
    {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2'},
    {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    {'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.5},
]

best_f1 = -1.0
best_config = None

for config in SEARCH_CONFIGS:
    model = RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        min_samples_split=config['min_samples_split'],
        min_samples_leaf=config['min_samples_leaf'],
        max_features=config['max_features'],
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )
    model.fit(data['X_train'], data['y_train'])
    validation_metrics = evaluate_binary_classifier(model, data['X_val'], data['y_val'])
    val_f1 = validation_metrics['f1']
    print(config, 'Validation F1 =', round(val_f1, 4))
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = config

print('Best configuration:', best_config)


{'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'} Validation F1 = 0.8809
{'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'} Validation F1 = 0.8991
{'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2'} Validation F1 = 0.8332
{'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'} Validation F1 = 0.9162
{'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.5} Validation F1 = 0.8282
Best configuration: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [8]:
final_model = RandomForestClassifier(
    n_estimators=best_config['n_estimators'],
    max_depth=best_config['max_depth'],
    min_samples_split=best_config['min_samples_split'],
    min_samples_leaf=best_config['min_samples_leaf'],
    max_features=best_config['max_features'],
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_trainval, y_trainval)

final_metrics = evaluate_binary_classifier(final_model, data['X_test'], data['y_test'])
print_metrics('Fine-Tuned Random Forest Results', final_metrics)



Fine-Tuned Random Forest Results
Accuracy : 0.9140
F1 Score : 0.9237
ROC-AUC  : 0.9779
Confusion Matrix:
  TN=2530  FP=207
  FN=345  TP=3340
              precision    recall  f1-score   support

     Focused       0.88      0.92      0.90      2737
  Distracted       0.94      0.91      0.92      3685

    accuracy                           0.91      6422
   macro avg       0.91      0.92      0.91      6422
weighted avg       0.92      0.91      0.91      6422



In [9]:
save_model_and_results(
    model=final_model,
    model_name='random_forest_finetuned.joblib',
    results_name='random_forest_finetuned_results.json',
    results={
        **final_metrics,
        'best_config': {
            **best_config,
            'max_depth': 'None' if best_config['max_depth'] is None else best_config['max_depth'],
        },
    },
)
